## APD Vision — Fase 8 Full Production (retraining gloves/shoes + gold gate)

Alur end-to-end di **Google Colab GPU** (Runtime → Change runtime type → GPU T4):

fetch 9 dataset (8 train + 1 gold) → `make_gold_test --split test` (gold beku) → `check_leak` (anti-bocor) → `merge_datasets` (train 4 kelas) → `copy_paste_augment` gloves/shoes → `train.py` → `promote_model.py` gate 4-kelas pada **gold** → unduh `best.pt` **hanya bila GATE LOLOS**.

**Strategi:** backbone full-PPE (label 4 kelas konsisten) + specialist gloves/shoes (angkat volume kelas lemah). GOLD = 1 dataset full-PPE ditahan penuh, test split jadi acuan.

**Anti-bocor:** GOLD beda sumber dari TRAIN, tak pernah masuk training; `check_leak` verifikasi via sha256.

**API key:** simpan di **Colab Secrets** (ikon 🔑) nama **`ROBOFLOW_API_KEY`**, aktifkan "Notebook access". Dibaca via `userdata` → env; tak pernah ditulis ke file.

**Lisensi:** hormati lisensi tiap dataset Roboflow. Edukasi/portfolio. **Jujur:** gate lolos/gagal = hasil empiris; kalau gagal lihat sel "Iterasi".

## 1. Setup — clone repo + dependensi + API key

In [ ]:
!nvidia-smi
%cd /content
!pip -q install ultralytics roboflow pyyaml

In [ ]:
# API key dari Colab Secrets (nama: ROBOFLOW_API_KEY) -> env. Tidak ditulis ke file.
from google.colab import userdata
import os
os.chdir('/content')
os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
print('ROBOFLOW_API_KEY termuat:', bool(os.environ.get('ROBOFLOW_API_KEY')))

## 2. Dataset: backbone full-PPE + specialist gloves/shoes

- **GOLD** (eval only, ditahan penuh): `ppe-yngjj/ppe-vum8g` — full 4 kelas, pakai test split-nya.
- **TRAIN backbone** (label 4 kelas konsisten): `constructionsite/ppe-fruwx`, `skcet-g4h72/construction-ppe-rdhzo`.
- **TRAIN specialist** (angkat volume kelas lemah): 3 gloves + 3 shoes.

**Jebakan single-class:** dataset gloves-only/shoes-only hanya melabeli satu kelas; objek PPE lain jadi *background* saat training → bisa turunkan recall helmet/vest. Karena itu gate (sel 6) memasang lantai untuk **keempat** kelas. Versi dikosongkan → ambil terbaru otomatis.

In [ ]:
import os
os.chdir('/content')

GOLD_SPEC = 'ppe-yngjj/ppe-vum8g'
TRAIN_SPECS = [
    # backbone full-PPE (4 kelas konsisten)
    'constructionsite/ppe-fruwx',
    'skcet-g4h72/construction-ppe-rdhzo',
    # specialist shoes
    'bns218/safety-shoes-yawyh',
    'parshvas-workspace-skr5a/safety-shoes-detection-uhdwv',
    'nedrick-chandra-gpg1l/safety-shoes-detection-5qgkg',
    # specialist gloves
    'tesis-n7wva/gloves-ppe',
    'aji-w7ttg/gloves-detection-lh2el',
    'test-pkubl/gloves-xjuja',
]
print('GOLD :', GOLD_SPEC)
print('TRAIN:', len(TRAIN_SPECS), 'dataset')

In [ ]:
# Unduh gold + semua train (env ROBOFLOW_API_KEY dibaca otomatis).
%cd /content
!python training/fetch_datasets.py --gold {GOLD_SPEC} --train {' '.join(TRAIN_SPECS)} --out-dir datasets
# Hasil: datasets/gold_src, datasets/train_0 .. datasets/train_7

## 3. Bekukan gold (test split) + cek anti-bocor terhadap semua train

In [ ]:
import glob, os
os.chdir('/content')
TRAIN_DIRS = sorted(glob.glob('datasets/train_*'))
print('train dirs:', TRAIN_DIRS)

!python training/make_gold_test.py --src datasets/gold_src --split test --out datasets/gold_prod
!python training/check_leak.py --gold datasets/gold_prod --train {' '.join(TRAIN_DIRS)}

## 4. Gabung semua train → 4 kelas + augmentasi gloves/shoes

In [ ]:
# Perhatikan peringatan 'kelas tak dikenal' di output: kalau ada varian gloves/shoes
# terlewat, tambahkan ke CLASS_MAP di training/merge_datasets.py lalu ulang sel ini.
%cd /content
!python training/merge_datasets.py --src {' '.join(TRAIN_DIRS)} --out datasets/ppe4
!python training/copy_paste_augment.py --data datasets/ppe4/data.yaml --classes gloves shoes --out datasets/ppe4_cp

## 5. Training (YOLO11s, imgsz 960). Jangan salin ke best.pt dulu — gate menentukan.

In [ ]:
%cd /content
!python training/train.py --data datasets/ppe4_cp/data.yaml --weights yolo11s.pt \
    --imgsz 960 --epochs 150 --name ppe4_prod
# candidate: runs/ppe4_prod/weights/best.pt

## 6. Gate promosi pada GOLD (lantai keempat kelas)

Lantai untuk **semua** kelas: `helmet=0.6 vest=0.6` menangkap regresi akibat jebakan single-class; `gloves=0.7 shoes=0.6` target kelas sulit. GATE LOLOS → salin ke `backend/weights/best.pt`. GATE GAGAL → exit 1, best.pt tak tersentuh.

In [ ]:
%cd /content
!python training/promote_model.py --candidate runs/ppe4_prod/weights/best.pt \
    --data datasets/gold_prod/data.yaml --imgsz 960 --min-map 0.45 \
    --min-class-map helmet=0.6 vest=0.6 gloves=0.7 shoes=0.6 \
    --json-out runs/gold_prod.json

## 7. Unduh best.pt (HANYA bila sel 6 = GATE LOLOS)

In [ ]:
from google.colab import files
import os
os.chdir('/content')
files.download('runs/ppe4_prod/weights/best.pt')
files.download('runs/gold_prod.json')
# Di lokal: taruh best.pt di backend/weights/, naikkan models.version di backend/config.yaml, commit.

## Iterasi bila GATE GAGAL

Catat `runs/gold_prod.json` apa adanya (jangan paksa lolos). Baca kelas mana yang jatuh:

- **helmet/vest turun** (jebakan single-class): buang 1–2 dataset gloves/shoes-only dari `TRAIN_SPECS`, atau tambah dataset full-PPE lain sebagai backbone.
- **gloves/shoes masih rendah:** (1) naikkan epoch `--epochs 200` + `--patience 30`; (2) ulang `copy_paste_augment` dua lapis atau tambah dataset specialist; (3) mulai dari bobot SH17 pretrained (`--weights sh17.pt`) ketimbang `yolo11s.pt` COCO.
- **Nama kelas ter-drop** (warning merge 'kelas tak dikenal'): tambahkan varian ke `CLASS_MAP` di `training/merge_datasets.py`, ulang sel 4.
- Ambang `--min-class-map` hanya diturunkan bila realistis untuk kasus pakai (shoes top-down memang sulit) — dokumentasikan, jangan diam-diam.